In [1]:
# ============================================================================
# PCA_Rollingtest.ipynb — Sprint C1 Research: Level Residuals
# ============================================================================
# Research question: Do PCA residuals on yield LEVELS produce better
# trading signals than PCA residuals on yield CHANGES?
#
# Key finding from cumulative residuals (abandoned):
#   Rolling cumulative sums of change residuals showed 5-14% both-pass
#   stationarity rates — not tradeable. Abandoned in favour of level approach.
#
# Level residuals approach (practitioner standard):
#   PCA on yield levels directly → residual = persistent yield mispricing
#   Cointegration ensures residuals are stationary despite non-stationary inputs
#
# Notebook structure:
#   Part 1: Baseline — daily change residuals (production)
#   Part 2: Level residuals with StandardScaler (wrong — kept as reference)
#   Part 3: Level residuals without StandardScaler (correct approach)
#   Part 4: Stationarity and signal quality analysis
#   Part 5: Future updates tracker
# ============================================================================

import sys
import warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, '.')

from data.fetch_rates import FetchRates
from analytics.pca import compute_pca_residuals
from analytics.stationarity import compute_rolling_adf, compute_rolling_kpss
from signals.zscore import compute_zscore
import config

print('Imports OK')
print(f'PCA_WINDOW={config.PCA_WINDOW} | ZSCORE_WINDOW={config.ZSCORE_WINDOW} | '
      f'Z_ENTRY_THRESHOLD=±{config.Z_ENTRY_THRESHOLD}')

Imports OK
PCA_WINDOW=60 | ZSCORE_WINDOW=252 | Z_ENTRY_THRESHOLD=±3.0


In [2]:
# ── Part 1: Baseline — daily change residuals (production approach) ─────────────────────────
rates_data   = FetchRates(config.START_DATE, mode=config.MODE)
residuals_df = compute_pca_residuals(rates_data, mode=config.MODE)
daily_zscore_df = compute_zscore(residuals_df, mode=config.MODE)

# Prepare yield levels for all level residual cells below
rates_df = rates_data.copy()
if 'Date' in rates_df.columns:
    rates_df = rates_df.set_index('Date')
rates_df.index = pd.to_datetime(rates_df.index)
yield_levels = rates_df[config.TENORS].dropna()

print(f'Rates loaded:         {len(rates_data)} trading days')
print(f'Change residuals:     {residuals_df.dropna().shape[0]} valid dates')
print(f'Date range:           {residuals_df.dropna().index[0].date()} → '
      f'{residuals_df.dropna().index[-1].date()}')
print(f'\nChange residual magnitude (std in bps):')
for tenor in config.TENORS:
    std_bps = residuals_df[tenor].dropna().std() * 10_000
    z3_bps  = std_bps * 3
    print(f'  {tenor:<8} std={std_bps:.4f} bps  |  Z=3.0 → {z3_bps:.4f} bps')
print(f'\nRound-trip cost: {config.TRANSACTION_COST_BPS * 2} bps')


Rolling PCA — vol-normalized yield levels
  Window=60d | n_components=2 | Mode=EOD
  Approach: level PCA with tenor-specific vol normalization
  Explained variance (sampled annually):
  [2006-03-01] PC1=75.1%  PC2=18.1%  Cumulative=93.2%
  [2007-03-02] PC1=83.6%  PC2=14.6%  Cumulative=98.2%
  [2008-03-04] PC1=79.5%  PC2=15.5%  Cumulative=95.0%
  [2009-03-06] PC1=85.0%  PC2=6.3%  Cumulative=91.4%
  [2010-03-10] PC1=59.7%  PC2=27.7%  Cumulative=87.4%
  [2011-03-10] PC1=60.3%  PC2=18.1%  Cumulative=78.4%
  [2012-03-13] PC1=54.5%  PC2=32.7%  Cumulative=87.1%
  [2013-03-15] PC1=55.5%  PC2=24.3%  Cumulative=79.8%
  [2014-03-19] PC1=61.0%  PC2=23.5%  Cumulative=84.6%
  [2015-03-23] PC1=69.5%  PC2=17.4%  Cumulative=86.9%
  [2016-03-23] PC1=72.1%  PC2=17.3%  Cumulative=89.4%
  [2017-03-27] PC1=78.8%  PC2=16.1%  Cumulative=94.9%
  [2018-03-28] PC1=88.2%  PC2=8.6%  Cumulative=96.8%
  [2019-04-02] PC1=75.5%  PC2=17.8%  Cumulative=93.3%
  [2020-04-03] PC1=95.9%  PC2=3.6%  Cumulative=99.5%
  [2021-

In [3]:
# ── PROD-A Verification: confirm vol-norm level residuals have correct magnitude ─
import config

# residuals_df is now computed by the updated compute_pca_residuals()
# using vol-normalized yield levels — not yield changes

print(f"{'='*65}")
print(f"PROD-A Gate 1 — Residual Magnitude Check")
print(f"Confirms pca.py is now using vol-norm level approach")
print(f"{'='*65}")
print(f"  {'Tenor':<8}  {'Std (bps)':>10}  {'Z=3 (bps)':>10}  "
      f"{'5x cost':>8}  {'Result':>8}")
print(f"  {'-'*52}")

cost_threshold = config.TRANSACTION_COST_BPS * 2 * 5  # 2.50 bps

for tenor in config.TENORS:
    std_bps = residuals_df[tenor].dropna().std() * 10_000
    z3_bps  = std_bps * 3
    result  = '✓ OK' if z3_bps > cost_threshold else '✗ FAIL'
    print(f"  {tenor:<8}  {std_bps:>9.2f}  {z3_bps:>9.2f}  "
          f"{cost_threshold:>7.2f}  {result:>8}")

print(f"{'='*65}")
print(f"5x cost threshold: {cost_threshold:.2f} bps")
print(f"Round-trip cost:   {config.TRANSACTION_COST_BPS * 2:.2f} bps")
print(f"\nLast 3 rows in bps (confirms level not change residuals):")
print((residuals_df.tail(3) * 10_000).round(2).to_string())

# Sanity check: change residuals had std ~0.0001-0.0005 bps
# Level residuals should have std ~2-9 bps
min_std = min(residuals_df[t].dropna().std() * 10_000 for t in config.TENORS)
max_std = max(residuals_df[t].dropna().std() * 10_000 for t in config.TENORS)
print(f"\nStd range across tenors: {min_std:.2f} — {max_std:.2f} bps")
print(f"Change residuals were:   0.01 — 0.09 bps")
print(f"Correctly using levels:  {'YES ✓' if min_std > 1.0 else 'NO ✗ — still using changes'}")

PROD-A Gate 1 — Residual Magnitude Check
Confirms pca.py is now using vol-norm level approach
  Tenor      Std (bps)   Z=3 (bps)   5x cost    Result
  ----------------------------------------------------
  1Mo            9.01      27.02     2.50      ✓ OK
  3Mo            4.39      13.18     2.50      ✓ OK
  6Mo            3.65      10.95     2.50      ✓ OK
  1Yr            3.75      11.26     2.50      ✓ OK
  2Yr            4.14      12.43     2.50      ✓ OK
  3Yr            3.68      11.05     2.50      ✓ OK
  5Yr            3.00       9.01     2.50      ✓ OK
  7Yr            3.18       9.53     2.50      ✓ OK
  10Yr           4.22      12.65     2.50      ✓ OK
  30Yr           7.16      21.48     2.50      ✓ OK
5x cost threshold: 2.50 bps
Round-trip cost:   0.50 bps

Last 3 rows in bps (confirms level not change residuals):
             1Mo   3Mo   6Mo   1Yr   2Yr   3Yr   5Yr   7Yr  10Yr  30Yr
Date                                                                  
2026-06-15 -3.27  3

In [57]:
# ── Part 2: Level residuals WITH StandardScaler — kept as reference ───────────────
# WHY THIS IS WRONG:
#   StandardScaler subtracts the rolling window mean before PCA.
#   Since yields drift from 0.36% (2020) to 4.38% (2024), the scaler
#   removes the level information PCA needs to capture cross-sectional
#   mispricing. What remains is essentially within-window yield variation
#   — close to change residuals computed via a more complex route.
#
# Evidence: Hurst exponents remained 0.80-0.86 regardless of scaler
#   → persistence is in the data, not introduced by preprocessing
#   → StandardScaler was not the root cause of the high H values
#
# This cell is kept for reference to document the investigation path.

PCA_WINDOW_L = config.PCA_WINDOW

level_resid_scaled = pd.DataFrame(
    index=yield_levels.index, columns=config.TENORS, dtype=float
)

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    for i in range(PCA_WINDOW_L, len(yield_levels)):
        fit_window   = yield_levels.iloc[i - PCA_WINDOW_L:i]
        current_obs  = yield_levels.iloc[i]
        current_date = yield_levels.index[i]
        try:
            scaler        = StandardScaler()
            scaled_window = scaler.fit_transform(fit_window)
            pca           = PCA(n_components=config.N_COMPONENTS)
            pca.fit(scaled_window)
            current_scaled    = scaler.transform(current_obs.values.reshape(1, -1))
            scores            = pca.transform(current_scaled)
            reconstructed_scaled = pca.inverse_transform(scores)
            reconstructed     = scaler.inverse_transform(reconstructed_scaled).flatten()
            level_resid_scaled.loc[current_date] = current_obs.values - reconstructed
        except Exception:
            level_resid_scaled.loc[current_date] = np.nan

level_resid_scaled = level_resid_scaled.astype(float)

print('Level residuals WITH StandardScaler — reference only')
print(f'Shape: {level_resid_scaled.dropna().shape}')
print(f'\nScaler mean drift example (5Yr rolling window mean over time):')
print('  This shows the scaler subtracts a different baseline each window')
sample_dates = pd.date_range('2008-01-01', '2026-01-01', freq='2YE')
print(f"  {'Date':<12}  {'5Yr Level':>10}  {'Scaler Mean':>12}  {'Scaled':>8}")
for date in sample_dates:
    if date not in yield_levels.index:
        continue
    idx = yield_levels.index.get_loc(date)
    if idx < PCA_WINDOW_L:
        continue
    fw  = yield_levels.iloc[idx - PCA_WINDOW_L:idx]
    sc  = StandardScaler().fit(fw)
    lv  = yield_levels.iloc[idx].iloc[config.TENORS.index('5Yr')]
    mn  = sc.mean_[config.TENORS.index('5Yr')]
    std = np.sqrt(sc.var_[config.TENORS.index('5Yr')])
    print(f'  {str(date.date()):<12}  {lv:>10.4f}  {mn:>12.4f}  '
          f'{(lv-mn)/std:>8.3f}')

Level residuals WITH StandardScaler — reference only
Shape: (5080, 10)

Scaler mean drift example (5Yr rolling window mean over time):
  This shows the scaler subtracts a different baseline each window
  Date           5Yr Level   Scaler Mean    Scaled
  2008-12-31        0.0155        0.0217    -1.161
  2010-12-31        0.0201        0.0150     1.431
  2012-12-31        0.0072        0.0069     0.509
  2014-12-31        0.0165        0.0160     0.577
  2018-12-31        0.0251        0.0289    -2.561
  2020-12-31        0.0036        0.0037    -0.333
  2024-12-31        0.0438        0.0413     1.265


In [58]:
# ── Part 3: Level residuals WITHOUT StandardScaler — correct approach ─────────────
# PCA is fit directly on raw yield levels within each 60d rolling window.
# No scaling — preserves the cross-sectional level relationship between tenors.
#
# Why no scaling is correct for level PCA:
#   Cointegration theory: the non-stationarity in individual yield levels
#   cancels out in PCA residuals because all yields share the same common
#   trend (Fed policy). StandardScaler destroys this by re-centering each
#   window, effectively differencing the data implicitly.
#
# Residual interpretation:
#   Positive residual = tenor yield ABOVE PCA fair value = CHEAP bond
#   Negative residual = tenor yield BELOW PCA fair value = RICH bond
#   Units: decimal yield → multiply by 10,000 to get bps

level_resid_noscale = pd.DataFrame(
    index=yield_levels.index, columns=config.TENORS, dtype=float
)
ev_log = []  # explained variance log

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    for i in range(PCA_WINDOW_L, len(yield_levels)):
        fit_window   = yield_levels.iloc[i - PCA_WINDOW_L:i].values
        current_obs  = yield_levels.iloc[i].values
        current_date = yield_levels.index[i]
        try:
            pca = PCA(n_components=config.N_COMPONENTS)
            pca.fit(fit_window)
            if (i - PCA_WINDOW_L) % 252 == 0:
                ev = pca.explained_variance_ratio_
                ev_log.append({'date': current_date,
                               'PC1': ev[0], 'PC2': ev[1],
                               'cumulative': ev[:config.N_COMPONENTS].sum()})
            scores        = pca.transform(current_obs.reshape(1, -1))
            reconstructed = pca.inverse_transform(scores).flatten()
            level_resid_noscale.loc[current_date] = current_obs - reconstructed
        except Exception:
            level_resid_noscale.loc[current_date] = np.nan

level_resid_noscale = level_resid_noscale.astype(float)
ev_df = pd.DataFrame(ev_log).set_index('date')

print('Level residuals WITHOUT StandardScaler — correct approach')
print(f'Shape: {level_resid_noscale.dropna().shape}')
print(f'\nMagnitude comparison (std in bps):')
print(f"  {'Tenor':<8}  {'Change std':>11}  {'Level std':>10}  "
      f"{'Z=3 bps (change)':>18}  {'Z=3 bps (level)':>17}")
print(f"  {'-'*68}")
for tenor in config.TENORS:
    c_std = residuals_df[tenor].dropna().std() * 10_000
    l_std = level_resid_noscale[tenor].dropna().std() * 10_000
    print(f'  {tenor:<8}  {c_std:>10.4f}  {l_std:>10.4f}  '
          f'{c_std*3:>18.2f}  {l_std*3:>17.2f}')
print(f'\nRound-trip cost: {config.TRANSACTION_COST_BPS * 2} bps')

print(f'\nExplained variance (annual snapshots):')
print(f"  {'Date':<12}  {'PC1':>8}  {'PC2':>8}  {'Cumul':>8}")
for date, row in ev_df.iterrows():
    print(f"  {str(date.date()):<12}  {row['PC1']*100:>7.1f}%  "
          f"{row['PC2']*100:>7.1f}%  {row['cumulative']*100:>7.1f}%")

Level residuals WITHOUT StandardScaler — correct approach
Shape: (5080, 10)

Magnitude comparison (std in bps):
  Tenor      Change std   Level std    Z=3 bps (change)    Z=3 bps (level)
  --------------------------------------------------------------------
  1Mo           4.1229      4.4904               12.37              13.47
  3Mo           2.1075      4.4888                6.32              13.47
  6Mo           1.7266      4.0989                5.18              12.30
  1Yr           1.8255      3.8704                5.48              11.61
  2Yr           1.9643      3.2791                5.89               9.84
  3Yr           1.6281      2.8153                4.88               8.45
  5Yr           1.1673      2.1716                3.50               6.51
  7Yr           1.2811      2.2926                3.84               6.88
  10Yr          1.6314      2.9328                4.89               8.80
  30Yr          2.7124      4.4937                8.14              13.48

R

In [59]:
# ── Part 4a: Full-sample ADF + KPSS on level residuals ───────────────────────────────
# Tests whether level residuals are stationary over the full 20-year sample.
# Cointegration theory predicts they should be — PCA extracts common trends
# (PC1=level shift, PC2=slope), leaving stationary idiosyncratic residuals.

print(f"{'='*75}")
print('Full-Sample Stationarity — Level Residuals (no-scale)')
print('ADF: p < 0.05 → stationary | KPSS: p > 0.05 → stationary')
print(f"{'='*75}")
print(f"  {'Tenor':<8}  {'ADF p':>8}  {'KPSS p':>8}  "
      f"{'ADF':>6}  {'KPSS':>6}  {'Verdict':>14}")
print(f"  {'-'*56}")

stationary_tenors = []
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    for tenor in config.TENORS:
        s      = level_resid_noscale[tenor].dropna()
        adf_p  = adfuller(s, autolag='AIC')[1]
        kpss_p = kpss(s, regression='c', nlags='auto')[1]
        ap     = adf_p  < 0.05
        kp     = kpss_p > 0.05
        verdict = ('STATIONARY'     if ap and kp  else
                   'NON-STATIONARY' if not ap and not kp else
                   'AMBIGUOUS')
        if verdict == 'STATIONARY':
            stationary_tenors.append(tenor)
        print(f"  {tenor:<8}  {adf_p:>8.4f}  {kpss_p:>8.4f}  "
              f"{'✓' if ap else '✗':>6}  {'✓' if kp else '✗':>6}  "
              f"{verdict:>14}")

print(f"{'='*75}")
print(f'Stationary: {stationary_tenors} ({len(stationary_tenors)}/10)')
print('Note: KPSS p-values bounded [0.01, 0.10] by statsmodels table')

Full-Sample Stationarity — Level Residuals (no-scale)
ADF: p < 0.05 → stationary | KPSS: p > 0.05 → stationary
  Tenor        ADF p    KPSS p     ADF    KPSS         Verdict
  --------------------------------------------------------
  1Mo         0.0000    0.0788       ✓       ✓      STATIONARY
  3Mo         0.0000    0.1000       ✓       ✓      STATIONARY
  6Mo         0.0000    0.0761       ✓       ✓      STATIONARY
  1Yr         0.0000    0.1000       ✓       ✓      STATIONARY
  2Yr         0.0000    0.1000       ✓       ✓      STATIONARY
  3Yr         0.0000    0.1000       ✓       ✓      STATIONARY
  5Yr         0.0000    0.0193       ✓       ✗       AMBIGUOUS
  7Yr         0.0000    0.0162       ✓       ✗       AMBIGUOUS
  10Yr        0.0000    0.1000       ✓       ✓      STATIONARY
  30Yr        0.0000    0.1000       ✓       ✓      STATIONARY
Stationary: ['1Mo', '3Mo', '6Mo', '1Yr', '2Yr', '3Yr', '10Yr', '30Yr'] (8/10)
Note: KPSS p-values bounded [0.01, 0.10] by statsmodels tab

In [60]:
# ── Part 4b: Hurst exponent — long memory test ────────────────────────────────────────────
# Run AFTER ADF+KPSS confirms stationarity.
# H < 0.5 → mean-reverting (anti-persistent) — faster than random walk
# H = 0.5 → random walk — standard ADF assumption
# H > 0.5 → trending/persistent — long memory, slow mean reversion
#
# For level residuals we found H = 0.80-0.86:
#   → NOT a problem — means dislocations persist 15-25 days before reverting
#   → This IS the tradeable signal duration
#   → Standard ADF needs 252d windows to detect stationarity (10+ cycles)

def hurst_rs(series, min_n=10):
    # Estimate Hurst exponent via R/S analysis.
    series = np.array(series.dropna())
    n      = len(series)
    rs_values, ns = [], []
    for chunk_size in [10, 20, 40, 60, 120, 252]:
        if chunk_size > n // 2:
            continue
        chunks    = [series[i:i+chunk_size]
                     for i in range(0, n - chunk_size + 1, chunk_size)]
        rs_chunk  = []
        for chunk in chunks:
            mean = np.mean(chunk)
            devs = np.cumsum(chunk - mean)
            r    = np.max(devs) - np.min(devs)
            s    = np.std(chunk, ddof=1)
            if s > 0:
                rs_chunk.append(r / s)
        if rs_chunk:
            rs_values.append(np.mean(rs_chunk))
            ns.append(chunk_size)
    if len(ns) < 2:
        return np.nan
    h = np.polyfit(np.log(ns), np.log(rs_values), 1)[0]
    return h

print(f"{'='*65}")
print('Hurst Exponent — Change vs Level Residuals')
print('H < 0.5 → mean-reverting | H = 0.5 → random walk | H > 0.5 → persistent')
print(f"{'='*65}")
print(f"  {'Tenor':<8}  {'H (change)':>12}  {'H (level)':>11}  {'Interpretation':>20}")
print(f"  {'-'*56}")

for tenor in config.TENORS:
    h_c = hurst_rs(residuals_df[tenor])
    h_l = hurst_rs(level_resid_noscale[tenor])
    interp = ('strong mean-reversion' if h_l < 0.4 else
              'mild mean-reversion'   if h_l < 0.5 else
              'near random walk'      if h_l < 0.55 else
              'long memory — slow MR' if h_l < 0.9 else
              'strongly persistent')
    print(f'  {tenor:<8}  {h_c:>12.3f}  {h_l:>11.3f}  {interp:>20}')

print(f"{'='*65}")
print('Level H=0.80-0.86 → long memory. ACF horizon 15-25 days confirms')
print('dislocations persist before reverting — this is the tradeable signal.')

Hurst Exponent — Change vs Level Residuals
H < 0.5 → mean-reverting | H = 0.5 → random walk | H > 0.5 → persistent
  Tenor       H (change)    H (level)        Interpretation
  --------------------------------------------------------
  1Mo              0.535        0.801  long memory — slow MR
  3Mo              0.472        0.821  long memory — slow MR
  6Mo              0.493        0.859  long memory — slow MR
  1Yr              0.501        0.837  long memory — slow MR
  2Yr              0.506        0.825  long memory — slow MR
  3Yr              0.529        0.843  long memory — slow MR
  5Yr              0.566        0.839  long memory — slow MR
  7Yr              0.524        0.839  long memory — slow MR
  10Yr             0.533        0.833  long memory — slow MR
  30Yr             0.553        0.828  long memory — slow MR
Level H=0.80-0.86 → long memory. ACF horizon 15-25 days confirms
dislocations persist before reverting — this is the tradeable signal.


In [61]:
# ── Part 4c: ACF comparison — mean reversion horizon ─────────────────────────────────
# ACF first crossing of 95% CI tells us the mean reversion horizon.
# Change residuals: 1-5 days  → short-term scalper
# Level residuals:  15-25 days → medium-term RV (practitioner standard)

MAX_LAGS     = 120  # 120 trading days ≈ 6 months
focus_tenors = ['2Yr', '5Yr', '10Yr']

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=(
        [f'{t} — Change residual ACF' for t in focus_tenors] +
        [f'{t} — Level residual ACF'  for t in focus_tenors]
    ),
    vertical_spacing=0.08, horizontal_spacing=0.08,
    column_titles=['Change residuals (current)', 'Level residuals (no-scale)']
)
colors = ['#1f77b4', '#2ca02c', '#d62728']

for row, tenor in enumerate(focus_tenors, 1):
    ci       = 1.96 / np.sqrt(len(residuals_df[tenor].dropna()))
    acf_c    = acf(residuals_df[tenor].dropna(),        nlags=MAX_LAGS, fft=True)
    acf_l    = acf(level_resid_noscale[tenor].dropna(), nlags=MAX_LAGS, fft=True)
    lags     = list(range(1, MAX_LAGS + 1))
    for col_idx, acf_vals in [(1, acf_c[1:]), (2, acf_l[1:])]:
        fig.add_trace(go.Bar(
            x=lags, y=acf_vals,
            marker_color=colors[row-1], showlegend=False
        ), row=row, col=col_idx)
        for y_val in [ci, -ci]:
            fig.add_hline(y=y_val, line_dash='dash', line_color='red',
                          line_width=1, opacity=0.5, row=row, col=col_idx)

fig.update_layout(
    title=dict(
        text='ACF Comparison: Change vs Level Residuals (120 lags ≈ 6 months)',
        x=0.5, font=dict(size=16)
    ),
    template='plotly_white', height=900
)
fig.show()

print(f"{'='*65}")
print('ACF First Crossing of 95% CI')
print(f"{'='*65}")
print(f"  {'Tenor':<8}  {'Change (days)':>14}  {'Level (days)':>14}")
print(f"  {'-'*40}")
for tenor in config.TENORS:
    ci    = 1.96 / np.sqrt(len(residuals_df[tenor].dropna()))
    acf_c = acf(residuals_df[tenor].dropna(),        nlags=MAX_LAGS, fft=True)[1:]
    acf_l = acf(level_resid_noscale[tenor].dropna(), nlags=MAX_LAGS, fft=True)[1:]
    c_cross = next((i+1 for i,v in enumerate(acf_c) if abs(v) < ci), f'>{MAX_LAGS}')
    l_cross = next((i+1 for i,v in enumerate(acf_l) if abs(v) < ci), f'>{MAX_LAGS}')
    print(f'  {tenor:<8}  {str(c_cross):>14}  {str(l_cross):>14}')
print(f"{'='*65}")
print('Level residuals: 15-25 day horizon → time stops should be 25-35 days')

ACF First Crossing of 95% CI
  Tenor      Change (days)    Level (days)
  ----------------------------------------
  1Mo                    4              15
  3Mo                    3              25
  6Mo                    4              21
  1Yr                    5              21
  2Yr                    2              15
  3Yr                    3              20
  5Yr                    2              21
  7Yr                    2              16
  10Yr                   1              18
  30Yr                   1              16
Level residuals: 15-25 day horizon → time stops should be 25-35 days


In [62]:
# ── Part 4d: Rolling stationarity — why longer windows are needed ─────────────────
# Level residuals take 15-25 days to mean-revert.
# ADF needs 5-10 full cycles to reliably reject unit root.
# → Need 120-252d windows (not 60d) for reliable stationarity detection.
#
# Results summary:
#   60d  window: all AVOID    (only 3 cycles — insufficient power)
#   120d window: BORDERLINE   (6 cycles — improving)
#   180d window: 5 TRADEABLE  (9 cycles — good)
#   252d window: 8 TRADEABLE  (12 cycles — reliable)
#
# Decision: use 252d rolling ADF+KPSS as entry gate for level residual strategy

for WINDOW in [60, 120, 180, 252]:
    print(f"\n{'='*65}")
    print(f'Window = {WINDOW}d (~{WINDOW//20} mean-reversion cycles)')
    print(f"{'='*65}")
    print(f"  {'Tenor':<8}  {'%ADF':>8}  {'%KPSS':>8}  "
          f"{'%Both':>8}  {'Verdict':>12}")
    print(f"  {'-'*50}")
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        for tenor in config.TENORS:
            series = level_resid_noscale[tenor].dropna()
            adf_r, kpss_r = [], []
            for i in range(WINDOW, len(series) + 1):
                w = series.iloc[i - WINDOW:i]
                try:
                    adf_r.append(adfuller(w, autolag='AIC')[1] < 0.05)
                    kpss_r.append(kpss(w, regression='c', nlags='auto')[1] > 0.05)
                except Exception:
                    adf_r.append(False)
                    kpss_r.append(False)
            pct_adf  = np.mean(adf_r)  * 100
            pct_kpss = np.mean(kpss_r) * 100
            pct_both = np.mean([a and k for a,k in zip(adf_r,kpss_r)]) * 100
            verdict  = ('TRADEABLE'  if pct_both >= 70 else
                        'BORDERLINE' if pct_both >= 50 else
                        'AVOID')
            print(f'  {tenor:<8}  {pct_adf:>7.1f}%  {pct_kpss:>7.1f}%  '
                  f'{pct_both:>7.1f}%  {verdict:>12}')


Window = 60d (~3 mean-reversion cycles)
  Tenor         %ADF     %KPSS     %Both       Verdict
  --------------------------------------------------
  1Mo          41.6%     71.0%     33.5%         AVOID
  3Mo          44.2%     67.5%     35.8%         AVOID
  6Mo          44.4%     62.7%     35.2%         AVOID
  1Yr          40.9%     63.8%     31.4%         AVOID
  2Yr          43.5%     70.5%     36.1%         AVOID
  3Yr          42.0%     66.7%     34.1%         AVOID
  5Yr          42.5%     66.8%     33.2%         AVOID
  7Yr          42.9%     66.1%     32.3%         AVOID
  10Yr         34.4%     60.7%     26.5%         AVOID
  30Yr         30.7%     61.7%     25.2%         AVOID

Window = 120d (~6 mean-reversion cycles)
  Tenor         %ADF     %KPSS     %Both       Verdict
  --------------------------------------------------
  1Mo          71.1%     77.4%     57.5%    BORDERLINE
  3Mo          69.6%     73.3%     55.3%    BORDERLINE
  6Mo          64.3%     67.2%     48.3% 

In [51]:
# ── Part 4e: Z-scores and signal magnitude verification ────────────────────────
# Normalization window: 252d (matches stationarity test window)
# Key question: at Z=3.0, how large is the actual residual in bps?
# Must exceed 0.50 bps round-trip cost to be economically viable.

ZSCORE_WINDOW_LEVEL = 252  # matches stationarity detection window

level_zscore_df = pd.DataFrame(index=level_resid_noscale.index)
for tenor in config.TENORS:
    s            = level_resid_noscale[tenor]
    roll_mean    = s.rolling(ZSCORE_WINDOW_LEVEL, min_periods=ZSCORE_WINDOW_LEVEL).mean()
    roll_std     = s.rolling(ZSCORE_WINDOW_LEVEL, min_periods=ZSCORE_WINDOW_LEVEL).std()
    level_zscore_df[tenor] = (s - roll_mean) / roll_std

print(f'Level Z-scores computed (normalization window: {ZSCORE_WINDOW_LEVEL}d)')
print(f'Valid dates: {level_zscore_df.dropna().shape[0]}')

print(f"\n{'='*75}")
print(f'Signal Magnitude at |Z| > {config.Z_ENTRY_THRESHOLD}')
print(f"{'='*75}")
print(f"  {'Tenor':<8}  {'N signals':>10}  {'Avg bps':>10}  "
      f"{'Max bps':>10}  {'vs cost':>10}")
print(f"  {'-'*55}")

cost_bps = config.TRANSACTION_COST_BPS * 2

for tenor in config.TENORS:
    z_s = level_zscore_df[tenor].dropna()
    r_s = (level_resid_noscale[tenor] * 10_000).reindex(z_s.index)
    mask = z_s.abs() > config.Z_ENTRY_THRESHOLD
    sig  = r_s[mask].abs()
    if len(sig) == 0:
        print(f"  {tenor:<8}  {'0':>10}")
        continue
    ratio = sig.mean() / cost_bps
    print(f'  {tenor:<8}  {len(sig):>10}  {sig.mean():>9.2f}  '
          f'{sig.max():>9.2f}  {ratio:>9.1f}x cost')

print(f"{'='*75}")
print(f'Round-trip cost: {cost_bps} bps')
print(f'Viable if avg bps >> {cost_bps} bps')

print(f'\nSignal frequency comparison:')
print(f"  {'Tenor':<8}  {'Change signals':>15}  {'Level signals':>14}")
print(f"  {'-'*42}")
for tenor in config.TENORS:
    d_sig = (daily_zscore_df[tenor].abs() > config.Z_ENTRY_THRESHOLD).sum()
    l_sig = (level_zscore_df[tenor].abs()  > config.Z_ENTRY_THRESHOLD).sum()
    print(f'  {tenor:<8}  {d_sig:>15}  {l_sig:>14}')

Level Z-scores computed (normalization window: 252d)
Valid dates: 4829

Signal Magnitude at |Z| > 3.0
  Tenor      N signals     Avg bps     Max bps     vs cost
  -------------------------------------------------------
  1Mo              116      12.90      35.15       25.8x cost
  3Mo              121      13.21      66.99       26.4x cost
  6Mo               96      10.44      46.15       20.9x cost
  1Yr              108      10.02      39.73       20.0x cost
  2Yr               71       9.04      23.35       18.1x cost
  3Yr               89       7.58      18.28       15.2x cost
  5Yr              121       6.37      16.66       12.7x cost
  7Yr               97       7.13      22.58       14.3x cost
  10Yr             100       8.39      29.00       16.8x cost
  30Yr             126      11.57      36.42       23.1x cost
Round-trip cost: 0.5 bps
Viable if avg bps >> 0.5 bps

Signal frequency comparison:
  Tenor      Change signals   Level signals
  -------------------------------

In [52]:
# ── Part 4f: Visual inspection ─────────────────────────────────────────────────────
# Plot level residuals in bps — confirm oscillation around zero
# and meaningful magnitude of excursions

level_resid_bps = level_resid_noscale * 10_000
colors_10 = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd',
             '#8c564b','#e377c2','#7f7f7f','#bcbd22','#17becf']

fig1 = make_subplots(
    rows=5, cols=2,
    subplot_titles=[f'{t} Level Residual (bps)' for t in config.TENORS],
    vertical_spacing=0.06, horizontal_spacing=0.08
)
for idx, tenor in enumerate(config.TENORS):
    row, col = idx // 2 + 1, idx % 2 + 1
    s = level_resid_bps[tenor].dropna()
    fig1.add_trace(go.Scatter(
        x=s.index, y=s.values, mode='lines',
        line=dict(width=1.0, color=colors_10[idx]), showlegend=False,
        hovertemplate=f'<b>{tenor}:</b> %{{y:.2f}} bps<extra></extra>'
    ), row=row, col=col)
    fig1.add_hline(y=0, line_dash='dash', line_color='gray',
                   line_width=1, opacity=0.4, row=row, col=col)
fig1.update_layout(
    title=dict(text='<b>PCA Level Residuals — All Tenors (bps)</b><br>'
               'Positive = CHEAP (yield above fair value) | '
               'Negative = RICH (yield below fair value)',
               x=0.5, font=dict(size=15)),
    template='plotly_white', height=1400
)
fig1.show()

focus = ['2Yr', '5Yr', '10Yr']
fig2  = make_subplots(rows=3, cols=1,
    subplot_titles=[f'{t} Level Z-score (252d normalization)' for t in focus],
    vertical_spacing=0.08)
for row, tenor in enumerate(focus, 1):
    z = level_zscore_df[tenor].dropna()
    fig2.add_trace(go.Scatter(
        x=z.index, y=z.values, mode='lines',
        line=dict(width=1.0, color=colors_10[config.TENORS.index(tenor)]),
        showlegend=False
    ), row=row, col=1)
    for y_val, c in [(config.Z_ENTRY_THRESHOLD, 'green'),
                     (-config.Z_ENTRY_THRESHOLD, 'red'), (0, 'gray')]:
        fig2.add_hline(y=y_val,
            line_dash='dash' if y_val != 0 else 'solid',
            line_color=c, line_width=1, opacity=0.5,
            row=row, col=1)
fig2.update_layout(
    title=dict(text=f'<b>Level Residual Z-scores — Belly Tenors</b><br>'
               f'Entry threshold ±{config.Z_ENTRY_THRESHOLD}',
               x=0.5, font=dict(size=15)),
    template='plotly_white', height=800
)
fig2.show()

In [53]:
# ── Part 5: Future updates tracker ──────────────────────────────────────────────────
# Living checklist — update status as each item is implemented

updates = {
    'C1-A: Rolling stationarity window': {
        'status': 'COMPLETE',
        'finding': '252d window needed — level residuals take 15-25 days to '
                   'mean-revert, requiring 10+ cycles for ADF to have power',
        'decision': 'Use 252d rolling ADF+KPSS as entry gate for level strategy'
    },
    'C1-B: Weak vs strict stationarity': {
        'status': 'CONFIRMED',
        'finding': 'Weak (covariance) stationarity is correct for trading. '
                   'ADF+KPSS test weak stationarity — sufficient for Z-scoring',
        'decision': 'No code change needed'
    },
    'C1-C: Johansen cointegration test': {
        'status': 'TODO',
        'finding': 'More appropriate than per-tenor ADF for confirming '
                   'cointegration rank. Should find rank=8 (10 yields - 2 PCs)',
        'decision': 'Add as new cell. Use as one-time full-sample validation'
    },
    'C1-D: Hurst exponent': {
        'status': 'COMPLETE',
        'finding': 'H=0.80-0.86 across all tenors. Long memory — dislocations '
                   'persist 15-25 days. NOT a problem, this is the signal.',
        'decision': 'Time stops: 25-35 days (ACF horizon + buffer)'
    },
    'C1-E: Structural break detection': {
        'status': 'TODO',
        'finding': 'Zivot-Andrews and Bai-Perron identify break dates. '
                   'Our rolling 60d gate catches breaks with 60d lag',
        'decision': 'Add as new cell. Run on full sample once for historical context'
    },
    'C1-F: Markov regime switching': {
        'status': 'TODO',
        'finding': '2-state Hamilton model gives P(high-vol regime) daily. '
                   'Suppress entries when P(regime 2) > 0.7',
        'decision': 'Low priority — implement after level residuals confirmed in backtest'
    },
    'C1-G: GARCH volatility regime': {
        'status': 'TODO',
        'finding': 'GARCH(1,1) captures volatility clustering in residuals. '
                   'Dynamic Z threshold in high-vol periods',
        'decision': 'Low priority — implement after Markov switching'
    },
    'C1-H: Error correction model': {
        'status': 'TODO',
        'finding': 'VECM identifies which tenors drive correction vs follow. '
                   'Speed of adjustment → regime-specific hold periods',
        'decision': 'Research only — not needed for production signal'
    },
    'PROD-A: Migrate level residuals to analytics/pca.py': {
        'status': 'PENDING — awaiting Cell 9 magnitude confirmation',
        'finding': 'Remove .diff() from compute_pca_residuals(). '
                   'No StandardScaler. Add YIELD_TO_BPS_SCALAR documentation.',
        'decision': 'Gate: Cell 9 avg bps > 5x round-trip cost for belly tenors'
    },
    'PROD-B: Update entry gate to 252d ADF+KPSS': {
        'status': 'PENDING — after PROD-A',
        'finding': 'Rolling 60d is too short for level residuals. '
                   'Need 252d window to reliably detect stationarity',
        'decision': 'Update ADF_ENTRY_WINDOW = 252 in config.py after PROD-A'
    },
    'PROD-C: Update time stops to 25-35 days': {
        'status': 'PENDING — after PROD-A',
        'finding': 'Current time stops based on ACF horizon of change residuals '
                   '(1-5 days). Level residuals need 25-35 day stops',
        'decision': 'Update TIME_STOP_BUFFER_MAP in config.py after PROD-A'
    },
    'PROD-D: IBKR feed integration': {
        'status': 'DEFERRED — Sprint D',
        'finding': 'OTR identification, EOD price pull, incremental FRED',
        'decision': 'After Sprint C fully validated'
    }
}

icon_map = {'COMPLETE': '✅', 'CONFIRMED': '✅', 'TODO': '⬜',
            'PENDING': '⏳', 'DEFERRED': '📋', 'IN PROGRESS': '🔄'}

print('=' * 70)
print('FUTURE UPDATES TRACKER — Sprint C1')
print('=' * 70)
for key, item in updates.items():
    status  = item['status']
    icon    = icon_map.get(status.split(' —')[0], '⬜')
    print(f'\n{icon} {key}')
    print(f"   Status:   {status}")
    print(f"   Finding:  {item['finding']}")
    print(f"   Decision: {item['decision']}")

FUTURE UPDATES TRACKER — Sprint C1

✅ C1-A: Rolling stationarity window
   Status:   COMPLETE
   Finding:  252d window needed — level residuals take 15-25 days to mean-revert, requiring 10+ cycles for ADF to have power
   Decision: Use 252d rolling ADF+KPSS as entry gate for level strategy

✅ C1-B: Weak vs strict stationarity
   Status:   CONFIRMED
   Finding:  Weak (covariance) stationarity is correct for trading. ADF+KPSS test weak stationarity — sufficient for Z-scoring
   Decision: No code change needed

⬜ C1-C: Johansen cointegration test
   Status:   TODO
   Finding:  More appropriate than per-tenor ADF for confirming cointegration rank. Should find rank=8 (10 yields - 2 PCs)
   Decision: Add as new cell. Use as one-time full-sample validation

✅ C1-D: Hurst exponent
   Status:   COMPLETE
   Finding:  H=0.80-0.86 across all tenors. Long memory — dislocations persist 15-25 days. NOT a problem, this is the signal.
   Decision: Time stops: 25-35 days (ACF horizon + buffer)

⬜ C1-E: 

In [54]:
# ============================================================================
# Cell 12 — Level Residuals: Variance Normalization Only (correct scaling)
# ============================================================================
# Problem with StandardScaler: subtracts window mean → removes level info
# Problem with no scaling:     high-vol tenors (1Mo, 30Yr) may dominate PCA
#                              in certain regimes (2008, 2022)
#
# Solution: divide each tenor by its OWN rolling window std only
#   scaled_tenor[t] = tenor_yield[t] / std(tenor_yield over fitting window)
#
# This equalizes volatility contribution across tenors while preserving
# the yield level that PCA needs to capture cross-sectional mispricing.
#
# Economic rationale:
#   Hiking cycle (2022): long-end vol >> short-end vol
#   Crisis (2008):       short-end vol >> long-end vol
#   Without equalization: PCA dominated by most volatile tenor in each regime
#   With vol-equalization: all tenors contribute equally to PC1/PC2 always
#
# Key difference from StandardScaler:
#   StandardScaler: scaled = (value - window_mean) / window_std  ← WRONG
#   Vol-norm:       scaled = value / window_std                   ← CORRECT
# ============================================================================

import warnings
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from statsmodels.tsa.stattools import adfuller, kpss
import config

PCA_WINDOW_L = config.PCA_WINDOW  # 60d

level_resid_volnorm = pd.DataFrame(
    index=yield_levels.index, columns=config.TENORS, dtype=float
)
ev_log_volnorm = []

with warnings.catch_warnings():
    warnings.simplefilter('ignore')

    for i in range(PCA_WINDOW_L, len(yield_levels)):
        fit_window   = yield_levels.iloc[i - PCA_WINDOW_L:i]
        current_obs  = yield_levels.iloc[i].values
        current_date = yield_levels.index[i]

        try:
            # Tenor-specific std from fitting window — NO mean subtraction
            window_stds = fit_window.std(axis=0).values
            window_stds = np.where(window_stds < 1e-10, 1e-10, window_stds)

            # Scale: divide by tenor std only — preserve level, equalize vol
            scaled_window  = fit_window.values  / window_stds
            current_scaled = current_obs        / window_stds

            # Fit PCA on vol-normalized yield levels
            pca = PCA(n_components=config.N_COMPONENTS)
            pca.fit(scaled_window)

            # Log explained variance annually
            if (i - PCA_WINDOW_L) % 252 == 0:
                ev = pca.explained_variance_ratio_
                ev_log_volnorm.append({
                    'date':       current_date,
                    'PC1':        ev[0],
                    'PC2':        ev[1],
                    'cumulative': ev[:config.N_COMPONENTS].sum()
                })

            # Project into PC space and reconstruct
            scores               = pca.transform(current_scaled.reshape(1, -1))
            reconstructed_scaled = pca.inverse_transform(scores).flatten()

            # Un-scale back to yield units before computing residual
            reconstructed = reconstructed_scaled * window_stds

            # Residual in yield level units (decimal)
            level_resid_volnorm.loc[current_date] = current_obs - reconstructed

        except Exception:
            level_resid_volnorm.loc[current_date] = np.nan

level_resid_volnorm = level_resid_volnorm.astype(float)
ev_df_volnorm       = pd.DataFrame(ev_log_volnorm).set_index('date')

# ── 1. Magnitude comparison ───────────────────────────────────────────────────
print(f"{'='*80}")
print(f'1. Magnitude Comparison — No-Scale vs Vol-Norm')
print(f'   Units: bps (×10,000) | Z=3.0 implied gross P&L vs '
      f'{config.TRANSACTION_COST_BPS * 2} bps round-trip cost')
print(f"{'='*80}")
print(f"  {'Tenor':<8}  {'NS std':>9}  {'VN std':>9}  "
      f"{'NS Z3 bps':>11}  {'VN Z3 bps':>11}  {'VN viable':>10}")
print(f"  {'-'*65}")
cost_bps = config.TRANSACTION_COST_BPS * 2
for tenor in config.TENORS:
    ns_std = level_resid_noscale[tenor].dropna().std() * 10_000
    vn_std = level_resid_volnorm[tenor].dropna().std() * 10_000
    vn_z3  = vn_std * 3
    viable = '✓' if vn_z3 > cost_bps * 5 else ('~' if vn_z3 > cost_bps else '✗')
    print(f'  {tenor:<8}  {ns_std:>8.2f}  {vn_std:>8.2f}  '
          f'{ns_std*3:>10.2f}  {vn_z3:>10.2f}  {viable:>10}')
print(f"  {'':8}  {'':9}  {'':9}  "
      f'  ✓ = Z3 > 5× cost ({cost_bps*5:.2f} bps)')

# ── 2. Explained variance ─────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f'2. Explained Variance — Vol-Norm (annual snapshots)')
print(f"{'='*60}")
print(f"  {'Date':<12}  {'PC1':>8}  {'PC2':>8}  {'Cumul':>8}")
for date, row in ev_df_volnorm.iterrows():
    flag = ' ← low' if row['cumulative'] < 0.75 else ''
    print(f"  {str(date.date()):<12}  {row['PC1']*100:>7.1f}%  "
          f"{row['PC2']*100:>7.1f}%  {row['cumulative']*100:>7.1f}%{flag}")

# ── 3. Full-sample stationarity ───────────────────────────────────────────────
print(f"\n{'='*75}")
print(f'3. Full-Sample Stationarity — Vol-Norm Level Residuals')
print(f'   ADF: p < 0.05 → stationary | KPSS: p > 0.05 → stationary')
print(f"{'='*75}")
print(f"  {'Tenor':<8}  {'ADF p':>8}  {'KPSS p':>8}  "
      f"{'ADF':>6}  {'KPSS':>6}  {'Verdict':>14}")
print(f"  {'-'*56}")
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    for tenor in config.TENORS:
        s      = level_resid_volnorm[tenor].dropna()
        adf_p  = adfuller(s, autolag='AIC')[1]
        kpss_p = kpss(s, regression='c', nlags='auto')[1]
        ap     = adf_p  < 0.05
        kp     = kpss_p > 0.05
        verdict = ('STATIONARY'     if ap and kp  else
                   'NON-STATIONARY' if not ap and not kp else
                   'AMBIGUOUS')
        print(f"  {tenor:<8}  {adf_p:>8.4f}  {kpss_p:>8.4f}  "
              f"{'✓' if ap else '✗':>6}  {'✓' if kp else '✗':>6}  "
              f"{verdict:>14}")

# ── 4. Hurst exponent ─────────────────────────────────────────────────────────
print(f"\n{'='*65}")
print(f'4. Hurst Exponent — No-Scale vs Vol-Norm')
print(f'   Target: H < 0.5 → mean-reverting')
print(f"{'='*65}")
print(f"  {'Tenor':<8}  {'H (no-scale)':>13}  {'H (vol-norm)':>13}  {'Change':>8}")
print(f"  {'-'*48}")
for tenor in config.TENORS:
    h_ns = hurst_rs(level_resid_noscale[tenor])
    h_vn = hurst_rs(level_resid_volnorm[tenor])
    delta = h_vn - h_ns
    flag  = '↓ better' if delta < -0.02 else ('↑ worse' if delta > 0.02 else '≈ same')
    print(f'  {tenor:<8}  {h_ns:>13.3f}  {h_vn:>13.3f}  {flag:>8}')

# ── 5. Rolling stationarity 252d comparison ───────────────────────────────────
print(f"\n{'='*70}")
print(f'5. Rolling Stationarity (252d) — No-Scale vs Vol-Norm')
print(f"{'='*70}")
print(f"  {'Tenor':<8}  {'%Both (NS)':>11}  {'%Both (VN)':>11}  "
      f"{'NS Verdict':>12}  {'VN Verdict':>12}")
print(f"  {'-'*60}")
WINDOW = 252
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    for tenor in config.TENORS:
        ns_both, vn_both = [], []
        for s, results in [
            (level_resid_noscale[tenor].dropna(), ns_both),
            (level_resid_volnorm[tenor].dropna(), vn_both)
        ]:
            for i in range(WINDOW, len(s) + 1):
                w = s.iloc[i - WINDOW:i]
                try:
                    ap = adfuller(w, autolag='AIC')[1]   < 0.05
                    kp = kpss(w, regression='c',
                              nlags='auto')[1]           > 0.05
                    results.append(ap and kp)
                except Exception:
                    results.append(False)
        pct_ns = np.mean(ns_both) * 100
        pct_vn = np.mean(vn_both) * 100
        def verdict(p):
            return 'TRADEABLE'  if p >= 70 else \
                   'BORDERLINE' if p >= 50 else 'AVOID'
        print(f'  {tenor:<8}  {pct_ns:>10.1f}%  {pct_vn:>10.1f}%  '
              f'{verdict(pct_ns):>12}  {verdict(pct_vn):>12}')

# ── 6. Signal magnitude at Z=3.0 ─────────────────────────────────────────────
print(f"\n{'='*70}")
print(f'6. Signal Magnitude — Vol-Norm at |Z| > {config.Z_ENTRY_THRESHOLD}')
print(f"{'='*70}")
vn_zscore_df = pd.DataFrame(index=level_resid_volnorm.index)
for tenor in config.TENORS:
    s         = level_resid_volnorm[tenor]
    roll_mean = s.rolling(252, min_periods=252).mean()
    roll_std  = s.rolling(252, min_periods=252).std()
    vn_zscore_df[tenor] = (s - roll_mean) / roll_std

print(f"  {'Tenor':<8}  {'N signals':>10}  {'Avg bps':>10}  "
      f"{'Max bps':>10}  {'x cost':>8}")
print(f"  {'-'*52}")
for tenor in config.TENORS:
    z_s  = vn_zscore_df[tenor].dropna()
    r_s  = (level_resid_volnorm[tenor] * 10_000).reindex(z_s.index)
    mask = z_s.abs() > config.Z_ENTRY_THRESHOLD
    sig  = r_s[mask].abs()
    if len(sig) == 0:
        print(f"  {tenor:<8}  {'0':>10}")
        continue
    ratio = sig.mean() / cost_bps
    print(f'  {tenor:<8}  {len(sig):>10}  {sig.mean():>9.2f}  '
          f'{sig.max():>9.2f}  {ratio:>7.1f}x')
print(f'\nRound-trip cost: {cost_bps} bps')

# ── 7. PROD-A decision gate ───────────────────────────────────────────────────
print(f"\n{'='*70}")
print(f'7. PROD-A Migration Decision Gate — Vol-Norm Approach')
print(f"{'='*70}")
belly   = ['2Yr', '3Yr', '5Yr', '7Yr', '10Yr']
all_ten = config.TENORS

# Gate 1: belly Z3 > 5x cost
belly_z3 = [
    level_resid_volnorm[t].dropna().std() * 10_000 * 3
    for t in belly
]
gate1 = all(z > cost_bps * 5 for z in belly_z3)

# Gate 2: rolling stationarity — count TRADEABLE/BORDERLINE at 252d
# (reuse vn_both from loop above — approximate via full-sample pass)
stat_pass = []
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    for tenor in all_ten:
        s     = level_resid_volnorm[tenor].dropna()
        adf_p = adfuller(s, autolag='AIC')[1]
        kss_p = kpss(s, regression='c', nlags='auto')[1]
        stat_pass.append(adf_p < 0.05 and kss_p > 0.05)
gate2 = sum(stat_pass) >= 7

print(f'  Gate 1 — Belly Z=3.0 > 5× cost ({cost_bps*5:.2f} bps): '
      f"{'✓ PASS' if gate1 else '✗ FAIL'}")
for t, z in zip(belly, belly_z3):
    print(f'    {t}: {z:.2f} bps')
print(f'\n  Gate 2 — Full-sample stationarity ≥7/10 tenors: '
      f"{'✓ PASS' if gate2 else '✗ FAIL'} "
      f'({sum(stat_pass)}/10 pass)')

if gate1 and gate2:
    print(f'\n  ✓ ALL GATES PASSED — proceed with PROD-A migration')
    print(f'    Replace compute_pca_residuals() with vol-norm level approach')
    print(f'    Update ADF_ENTRY_WINDOW = 252 in config.py')
    print(f'    Update TIME_STOP_BUFFER_MAP for 25-35 day holds')
else:
    print(f'\n  ✗ GATES NOT PASSED — do not migrate yet')
    print(f'    Review failed gates above before proceeding')

1. Magnitude Comparison — No-Scale vs Vol-Norm
   Units: bps (×10,000) | Z=3.0 implied gross P&L vs 0.5 bps round-trip cost
  Tenor        NS std     VN std    NS Z3 bps    VN Z3 bps   VN viable
  -----------------------------------------------------------------
  1Mo           4.49      9.01       13.47       27.02           ✓
  3Mo           4.49      4.39       13.47       13.18           ✓
  6Mo           4.10      3.65       12.30       10.95           ✓
  1Yr           3.87      3.75       11.61       11.26           ✓
  2Yr           3.28      4.14        9.84       12.43           ✓
  3Yr           2.82      3.68        8.45       11.05           ✓
  5Yr           2.17      3.00        6.51        9.01           ✓
  7Yr           2.29      3.18        6.88        9.53           ✓
  10Yr          2.93      4.22        8.80       12.65           ✓
  30Yr          4.49      7.16       13.48       21.48           ✓
                                    ✓ = Z3 > 5× cost (2.50 bps)

2.

In [63]:
# ============================================================================
# Cell 13 — Rolling Stationarity Voting: 120d, 180d, 252d Windows
# ============================================================================
# Purpose: evaluate whether a multi-window voting system makes sense
# for vol-norm level residuals, similar to the 10/15/20/60d system
# built for change residuals.
#
# Key question: do 120d and 180d windows add early-warning value
# over 252d alone, given level residuals take 15-25 days to mean-revert?
#
# For each window we compute:
#   1. %Both Pass (ADF+KPSS) — stationarity reliability
#   2. Signal frequency — how many tradeable signals exist
#   3. Voting agreement — do windows agree or disagree
#
# Windows tested: 120d (~6 MR cycles), 180d (~9 cycles), 252d (~12 cycles)
# ============================================================================

import warnings
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import adfuller, kpss
import config

WINDOWS = [120, 180, 252]

# ── Step 1: Compute rolling ADF+KPSS pass/fail for each window ───────────────
# Store as dict: {window: DataFrame of bool (date × tenor)}
rolling_stat = {}

with warnings.catch_warnings():
    warnings.simplefilter('ignore')

    for WINDOW in WINDOWS:
        print(f'Computing {WINDOW}d rolling stationarity...', end=' ')
        results = {}

        for tenor in config.TENORS:
            series    = level_resid_volnorm[tenor].dropna()
            both_pass = []
            dates     = []

            for i in range(WINDOW, len(series) + 1):
                w = series.iloc[i - WINDOW:i]
                try:
                    ap = adfuller(w, autolag='AIC')[1] < 0.05
                    kp = kpss(w, regression='c', nlags='auto')[1] > 0.05
                    both_pass.append(ap and kp)
                except Exception:
                    both_pass.append(False)
                dates.append(series.index[i - 1])

            results[tenor] = pd.Series(both_pass, index=dates, dtype=bool)

        rolling_stat[WINDOW] = pd.DataFrame(results)
        print(f'done — {len(rolling_stat[WINDOW])} dates')

# ── Step 2: Summary table — %Both Pass per window ────────────────────────────
print(f"\n{'='*80}")
print(f'Rolling Stationarity %Both Pass — Vol-Norm Level Residuals')
print(f"{'='*80}")
header = f"  {'Tenor':<8}"
for w in WINDOWS:
    header += f"  {str(w)+'d %Both':>12}"
header += f"  {'120 Verdict':>12}  {'180 Verdict':>12}  {'252 Verdict':>12}"
print(header)
print(f"  {'-'*75}")

def verdict(p):
    return 'TRADEABLE'  if p >= 70 else \
           'BORDERLINE' if p >= 50 else 'AVOID'

window_verdicts = {w: {} for w in WINDOWS}

for tenor in config.TENORS:
    row = f'  {tenor:<8}'
    for w in WINDOWS:
        df  = rolling_stat[w]
        if tenor not in df.columns:
            row += f"  {'N/A':>12}"
            continue
        pct = df[tenor].mean() * 100
        window_verdicts[w][tenor] = verdict(pct)
        row += f'  {pct:>11.1f}%'
    for w in WINDOWS:
        row += f"  {window_verdicts[w].get(tenor,'N/A'):>12}"
    print(row)

print(f'\n  Summary:')
for w in WINDOWS:
    n_trade  = sum(1 for v in window_verdicts[w].values() if v == 'TRADEABLE')
    n_border = sum(1 for v in window_verdicts[w].values() if v == 'BORDERLINE')
    n_avoid  = sum(1 for v in window_verdicts[w].values() if v == 'AVOID')
    print(f'  {w}d: {n_trade} TRADEABLE  {n_border} BORDERLINE  {n_avoid} AVOID')

# ── Step 3: Voting agreement analysis ────────────────────────────────────────
# On dates where all three windows have data, do they agree?
# High agreement → voting adds little (252d is sufficient)
# Low agreement  → voting adds value (windows detect different things)

print(f"\n{'='*80}")
print(f'Voting Agreement Analysis — Do Windows Agree?')
print(f'On dates where all 3 windows have data')
print(f"{'='*80}")
print(f"  {'Tenor':<8}  {'All agree':>10}  {'2/3 agree':>10}  "
      f"{'Disagree':>10}  {'120≠252':>10}  {'180≠252':>10}")
print(f"  {'-'*60}")

for tenor in config.TENORS:
    common = (rolling_stat[120].index
              .intersection(rolling_stat[180].index)
              .intersection(rolling_stat[252].index))

    if tenor not in rolling_stat[120].columns:
        continue

    v120 = rolling_stat[120].loc[common, tenor]
    v180 = rolling_stat[180].loc[common, tenor]
    v252 = rolling_stat[252].loc[common, tenor]

    all_agree    = ((v120 == v180) & (v180 == v252)).mean() * 100
    two_agree    = (((v120 == v180) | (v180 == v252) | (v120 == v252)) &
                   ~((v120 == v180) & (v180 == v252))).mean() * 100
    all_disagree = 100 - all_agree - two_agree

    diff_120_252 = (v120 != v252).mean() * 100
    diff_180_252 = (v180 != v252).mean() * 100

    print(f'  {tenor:<8}  {all_agree:>9.1f}%  {two_agree:>9.1f}%  '
          f'{all_disagree:>9.1f}%  {diff_120_252:>9.1f}%  {diff_180_252:>9.1f}%')

# ── Step 4: Early warning test ────────────────────────────────────────────────
# When 252d flips non-stationary, does 120d or 180d detect it earlier?
# This tells us if shorter windows add early-warning value.

print(f"\n{'='*80}")
print(f'Early Warning Test — Does 120d/180d Detect Regime Breaks Before 252d?')
print(f'Looking for: 252d flips False while 120d/180d was already False')
print(f"{'='*80}")
print(f"  {'Tenor':<8}  {'252d breaks':>12}  {'120d early':>11}  "
      f"{'180d early':>11}  {'120d lag':>10}  {'180d lag':>10}")
print(f"  {'-'*65}")

for tenor in config.TENORS:
    if tenor not in rolling_stat[252].columns:
        continue

    common = (rolling_stat[120].index
              .intersection(rolling_stat[180].index)
              .intersection(rolling_stat[252].index))

    v120 = rolling_stat[120].loc[common, tenor].astype(int)
    v180 = rolling_stat[180].loc[common, tenor].astype(int)
    v252 = rolling_stat[252].loc[common, tenor].astype(int)

    breaks_252 = (v252.diff() == -1)
    n_breaks   = breaks_252.sum()

    if n_breaks == 0:
        print(f"  {tenor:<8}  {'0':>12}")
        continue

    early_120, early_180 = 0, 0
    lag_120_list, lag_180_list = [], []

    for break_date in common[breaks_252.values]:
        lookback = common[common < break_date][-60:]
        if len(lookback) == 0:
            continue

        pre_120 = v120.loc[lookback]
        pre_180 = v180.loc[lookback]

        false_120 = pre_120[pre_120 == 0]
        false_180 = pre_180[pre_180 == 0]

        if len(false_120) > 0:
            early_120 += 1
            lag_days = (break_date - false_120.index[-1]).days
            lag_120_list.append(lag_days)

        if len(false_180) > 0:
            early_180 += 1
            lag_days = (break_date - false_180.index[-1]).days
            lag_180_list.append(lag_days)

    avg_lag_120 = np.mean(lag_120_list) if lag_120_list else 0
    avg_lag_180 = np.mean(lag_180_list) if lag_180_list else 0

    print(f'  {tenor:<8}  {n_breaks:>12}  {early_120:>11}  '
          f'{early_180:>11}  {avg_lag_120:>9.1f}d  {avg_lag_180:>9.1f}d')

print(f'\n  Interpretation:')
print(f"  '120d early' = how many 252d breaks were preceded by 120d going False")
print(f"  '120d lag'   = avg days before 252d break that 120d already flagged it")
print(f'  Large lag → 120d/180d add early warning value → voting system useful')
print(f'  Small lag → 252d is fast enough → single window sufficient')

# ── Step 5: Voting system design recommendation ───────────────────────────────
print(f"\n{'='*80}")
print(f'Voting System Design — Recommendation')
print(f"{'='*80}")
print(f"""
Based on the analysis above, the voting system for level residuals should:

Option A — Single window (252d only):
  Use if: 120d/180d disagree rarely with 252d (low 120≠252 %)
           AND early warning lag is small (<10 days)
  Rationale: 252d is the only statistically reliable window.
             Shorter windows add noise not signal.

Option B — Multi-window voting (120d + 180d + 252d):
  Use if: 120d/180d frequently disagree with 252d (high 120≠252 %)
           AND early warning lag is meaningful (>10 days before 252d)
  Rationale: shorter windows catch regime breaks earlier,
             worth accepting some false positives for earlier exit.

Option C — Asymmetric gates:
  Entry:  252d only (most reliable, fewest false blocks)
  Exit:   180d + 252d majority vote (earlier exit on regime breaks)
  Rationale: same design principle as change residual strategy —
             different windows serve different purposes.
""")

Computing 120d rolling stationarity... done — 4961 dates
Computing 180d rolling stationarity... done — 4901 dates
Computing 252d rolling stationarity... done — 4829 dates

Rolling Stationarity %Both Pass — Vol-Norm Level Residuals
  Tenor       120d %Both    180d %Both    252d %Both   120 Verdict   180 Verdict   252 Verdict
  ---------------------------------------------------------------------------
  1Mo              53.9%         68.8%         80.0%    BORDERLINE    BORDERLINE     TRADEABLE
  3Mo              63.0%         75.3%         78.9%    BORDERLINE     TRADEABLE     TRADEABLE
  6Mo              57.1%         76.8%         80.5%    BORDERLINE     TRADEABLE     TRADEABLE
  1Yr              57.0%         70.9%         79.3%    BORDERLINE     TRADEABLE     TRADEABLE
  2Yr              55.9%         77.8%         84.0%    BORDERLINE     TRADEABLE     TRADEABLE
  3Yr              47.7%         62.3%         70.5%         AVOID    BORDERLINE     TRADEABLE
  5Yr              49.0%  

In [ ]:
# ============================================================================
# Cell 15 — Backtest Diagnostic Investigations — Level Residual Strategy
# ============================================================================
# Three investigations after first successful level residual backtest:
#   1. Zero hold trades  — potential same-day entry/exit logic issue
#   2. Stop-loss trades  — 6.3% rate, check tenor/year concentration
#   3. Flat year analysis — 2014 and 2021 low-signal years confirmed
#
# Self-contained: runs full pipeline if backtest_results not already in memory.
# ============================================================================

import pandas as pd
import numpy as np
import sys, os

# ── Ensure project root on path ───────────────────────────────────────────────
_proj_root = os.path.abspath('')
if _proj_root not in sys.path:
    sys.path.insert(0, _proj_root)

import config
from data.fetch_rates import FetchRates
from analytics.pca import compute_pca_residuals
from analytics.stationarity import compute_rolling_adf, compute_rolling_kpss, compute_acf_summary
from signals.zscore import compute_zscore
from backtest.engine import run_backtest

# ── Populate backtest_results if not in memory ────────────────────────────────
try:
    backtest_results
    print("backtest_results already in memory — skipping pipeline run")
except NameError:
    print("backtest_results not found — running full pipeline...")
    rates_data = FetchRates(start_date=config.START_DATE, mode='EOD')
    print(f"  Fetched {len(rates_data)} trading days")
    residuals_df = compute_pca_residuals(
        rates_data, window=config.PCA_WINDOW, n_components=config.N_COMPONENTS
    )
    print(f"  Residuals: {residuals_df.dropna().shape[0]} valid dates")
    rolling_adfs      = compute_rolling_adf(residuals_df, windows=config.ROLLING_ADF_WINDOWS)
    rolling_adf_252d  = rolling_adfs[252]
    rolling_adf_180d  = rolling_adfs[180]
    rolling_kpss_252d = compute_rolling_kpss(residuals_df, window=252)
    rolling_kpss_180d = compute_rolling_kpss(residuals_df, window=180)
    z_score_df        = compute_zscore(residuals_df, window=config.ZSCORE_WINDOW)
    acf_summary_df    = compute_acf_summary(residuals_df)
    _no_auction       = lambda date, tenor, auction_calendar, mode=None: ('CLEAR', {})
    backtest_results  = run_backtest(
        residuals_df=residuals_df,
        z_score_df=z_score_df,
        rolling_adf_252d=rolling_adf_252d,
        rolling_kpss_252d=rolling_kpss_252d,
        rolling_adf_180d=rolling_adf_180d,
        rolling_kpss_180d=rolling_kpss_180d,
        rolling_adfs=rolling_adfs,
        acf_summary_df=acf_summary_df,
        auction_calendar=pd.DataFrame(),
        get_auction_flag_fn=_no_auction,
    )
    print("  Pipeline complete — backtest_results populated\n")

# ── Build trade log DataFrame ─────────────────────────────────────────────────
trade_log_df = pd.DataFrame(backtest_results['trade_log'])
trade_log_df['entry_date'] = pd.to_datetime(trade_log_df['entry_date'])
trade_log_df['exit_date']  = pd.to_datetime(trade_log_df['exit_date'])
trade_log_df['year']       = trade_log_df['entry_date'].dt.year

print(f"Total trades in log: {len(trade_log_df)}")
print(f"Columns: {list(trade_log_df.columns)}")

# ── Investigation 1: Zero hold trades ────────────────────────────────────────
print(f"\n{'='*70}")
print(f"Investigation 1 — Zero Hold Trades")
print(f"Expected: 0 (is_entry_day guard should prevent same-day exit)")
print(f"{'='*70}")

zero_hold = trade_log_df[trade_log_df['hold_days'] == 0]
print(f"Zero hold trade count: {len(zero_hold)}")

if len(zero_hold) > 0:
    print(f"\nDetail:")
    cols = ['tenor', 'direction', 'entry_date', 'exit_date',
            'exit_reason', 'entry_zscore', 'exit_zscore', 'pnl_bps']
    available = [c for c in cols if c in zero_hold.columns]
    print(zero_hold[available].to_string())
    print(f"\nZero hold by exit reason:")
    print(zero_hold['exit_reason'].value_counts().to_string())
    print(f"\nZero hold by tenor:")
    print(zero_hold['tenor'].value_counts().to_string())
else:
    print("✓ No zero hold trades — is_entry_day guard working correctly")

# ── Investigation 2: Stop-loss trades ────────────────────────────────────────
print(f"\n{'='*70}")
print(f"Investigation 2 — Stop-Loss Trades (6.3% rate = 18 trades)")
print(f"Stop fires when Z > entry_Z + 2.0 (LONG) or Z < entry_Z - 2.0 (SHORT)")
print(f"{'='*70}")

sl_trades = trade_log_df[trade_log_df['exit_reason'] == 'STOP-LOSS']
print(f"Stop-loss trade count: {len(sl_trades)}")

if len(sl_trades) > 0:
    cols = ['tenor', 'direction', 'entry_date', 'exit_date',
            'entry_zscore', 'exit_zscore', 'hold_days', 'pnl_bps']
    available = [c for c in cols if c in sl_trades.columns]
    print(f"\nFull stop-loss trade list:")
    print(sl_trades[available].to_string())

    print(f"\nStop-loss by tenor:")
    sl_by_tenor = sl_trades.groupby('tenor').agg(
        n=('pnl_bps', 'count'),
        avg_pnl=('pnl_bps', 'mean'),
        total_pnl=('pnl_bps', 'sum')
    ).round(2)
    print(sl_by_tenor.to_string())

    print(f"\nStop-loss by year:")
    sl_by_year = sl_trades.groupby('year').agg(
        n=('pnl_bps', 'count'),
        avg_pnl=('pnl_bps', 'mean'),
        total_pnl=('pnl_bps', 'sum')
    ).round(2)
    print(sl_by_year.to_string())

    print(f"\nStop-loss by direction:")
    print(sl_trades['direction'].value_counts().to_string())

    print(f"\nEntry Z-score distribution for stop-loss trades:")
    if 'entry_zscore' in sl_trades.columns:
        print(f"  Mean entry |Z|:  {sl_trades['entry_zscore'].abs().mean():.3f}")
        print(f"  Min entry |Z|:   {sl_trades['entry_zscore'].abs().min():.3f}")
        print(f"  Max entry |Z|:   {sl_trades['entry_zscore'].abs().max():.3f}")
        print(f"  Entries near threshold (|Z| < 3.2): "
              f"{(sl_trades['entry_zscore'].abs() < 3.2).sum()} trades")

    sl_cost    = sl_trades['pnl_bps'].sum()
    total_pnl  = trade_log_df['pnl_bps'].sum()
    print(f"\nP&L impact:")
    print(f"  Stop-loss total P&L: {sl_cost:.2f} bps")
    print(f"  As % of total net:   {sl_cost/total_pnl*100:.1f}%")
    print(f"  Avg loss per stop:   {sl_cost/len(sl_trades):.2f} bps")

# ── Investigation 3: Year-by-year analysis ───────────────────────────────────
print(f"\n{'='*70}")
print(f"Investigation 3 — Year-by-Year P&L Analysis")
print(f"Focus: confirm 2014 and 2021 are low-signal, not data issues")
print(f"{'='*70}")

by_year = trade_log_df.groupby('year').agg(
    n_trades=('pnl_bps', 'count'),
    gross_bps=('pnl_bps', lambda x: x.sum() + len(x) * config.TRANSACTION_COST_BPS * 2),
    net_bps=('pnl_bps', 'sum'),
    win_rate=('pnl_bps', lambda x: (x > 0).mean() * 100),
    avg_hold=('hold_days', 'mean'),
    mr_exits=('exit_reason', lambda x: (x == 'MEAN-REVERSION').sum()),
    sl_exits=('exit_reason', lambda x: (x == 'STOP-LOSS').sum()),
    ns_exits=('exit_reason', lambda x: (x == 'NON-STATIONARY').sum()),
).round(2)

print(f"\n{'Year':<6} {'N':>5} {'Gross':>8} {'Net':>8} {'Win%':>7} "
      f"{'AvgHold':>8} {'MR':>5} {'SL':>5} {'NS':>5}")
print(f"  {'-'*62}")
for year, row in by_year.iterrows():
    flag = ''
    if row['n_trades'] <= 5:
        flag = ' ← low signal'
    elif row['net_bps'] < 0:
        flag = ' ← loss year'
    print(f"  {year:<6} {row['n_trades']:>5.0f} {row['gross_bps']:>8.2f} "
          f"{row['net_bps']:>8.2f} {row['win_rate']:>6.1f}% "
          f"{row['avg_hold']:>7.1f}d {row['mr_exits']:>5.0f} "
          f"{row['sl_exits']:>5.0f} {row['ns_exits']:>5.0f}{flag}")

print(f"\n2014 detail:")
y2014 = trade_log_df[trade_log_df['year'] == 2014]
if len(y2014) > 0:
    cols = ['tenor', 'direction', 'entry_date', 'exit_reason', 'pnl_bps']
    available = [c for c in cols if c in y2014.columns]
    print(y2014[available].to_string())
else:
    print("  No trades in 2014")

print(f"\n2021 detail:")
y2021 = trade_log_df[trade_log_df['year'] == 2021]
if len(y2021) > 0:
    cols = ['tenor', 'direction', 'entry_date', 'exit_reason', 'pnl_bps']
    available = [c for c in cols if c in y2021.columns]
    print(y2021[available].to_string())
else:
    print("  No trades in 2021")

# ── Summary verdict ───────────────────────────────────────────────────────────
print(f"\n{'='*70}")
print(f"Diagnostic Summary")
print(f"{'='*70}")
print(f"  Zero hold trades:   {len(zero_hold)} "
      f"{'✓ clean' if len(zero_hold) == 0 else '✗ investigate'}")
print(f"  Stop-loss trades:   {len(sl_trades)} ({len(sl_trades)/len(trade_log_df)*100:.1f}%)")
if len(sl_trades) > 0:
    print(f"  Stop-loss P&L drag: {sl_trades['pnl_bps'].sum():.2f} bps")
print(f"  Loss years:         {(by_year['net_bps'] < 0).sum()} / {len(by_year)}")
print(f"  Low-signal years:   {(by_year['n_trades'] <= 5).sum()} (<=5 trades)")
